# Notebook 1 — Bronze Layer Ingestion

**Project:** Tractor Industry Analytics  
**Layer:** Bronze (raw ingestion → Delta tables)  
**Author:** Tractor Analytics Pipeline  
**Last Updated:** 2026-06-18

## Overview
This notebook ingests all raw CSV sources into the `bronze` schema as Delta tables.
No transformations are applied — data is stored as-is with ingestion metadata appended.

## Tables Created
| Table | Source File | Description |
|---|---|---|
| `bronze.india_tractor_sales` | india_tractor_sales_2004_2025.csv | India annual tractor sales by fiscal year (TMA) |
| `bronze.india_tractor_seasonality` | india_tractor_monthly_seasonality.csv | Monthly sales share with Kharif/Rabi season labels |
| `bronze.india_tractor_brands` | india_tractor_brand_market_share.csv | India brand-level market share data |
| `bronze.us_aem_tractor_sales_hp` | us_aem_tractor_sales_by_hp.csv | US retail tractor sales split by HP category (AEM) |
| `bronze.global_tractor_production` | global_tractor_production_by_country.csv | Country-level production, export, and domestic sales |
| `bronze.comtrade_hs8701_exports` | comtrade_hs8701_tractors_2015_2022.csv | UN Comtrade bilateral trade flows for HS code 8701 |
| `bronze.fred_farm_equipment_ppi` | fred_wpu114.csv | FRED WPU114 — Farm & Garden Machinery PPI (monthly) |
| `bronze.fred_farm_equip_ppi_detail` | fred_farm_equipment_ppi.csv | FRED WPUFD41312 — Farm Equipment PPI detail (monthly) |
| `bronze.fred_corn_price_index` | fred_corn_price_index.csv | FRED WPU012202 — Corn PPI (monthly) |
| `bronze.fred_wheat_price_index` | fred_wheat_price_index.csv | FRED WPU012101 — Wheat PPI (monthly) |
| `bronze.fred_us_net_farm_income` | fred_us_net_farm_income.csv | FRED A368RC1A027NBEA — US Net Farm Income (annual) |
| `bronze.faostat_arable_land` | faostat_arable_land_selected.csv | FAOSTAT arable land area (1000 ha) by country/year |

## Prerequisites
1. Upload all CSVs from `data/raw/` to DBFS at `/FileStore/tractor/raw/`:  
   **Databricks UI → Data → Add data → Upload files → `/FileStore/tractor/raw/`**
2. Cluster: Databricks Runtime 13.x or higher (supports Delta Lake 2.x)
3. No external libraries required — uses built-in PySpark + Delta


## Cell 1 — Environment Setup
Detect whether we are running on Databricks or locally (for development/testing).

In [ ]:
import os
import sys
from datetime import datetime, timezone

# ── Runtime detection ───────────────────────────────────────────────────────
try:
    dbutils  # noqa: F821 — defined only inside Databricks
    ON_DATABRICKS = True
except NameError:
    ON_DATABRICKS = False

print(f"Running on Databricks: {ON_DATABRICKS}")

# ── Paths ────────────────────────────────────────────────────────────────────
if ON_DATABRICKS:
    RAW_PATH = "/FileStore/tractor/raw"          # DBFS path (omit dbfs: prefix for spark.read)
    DBFS_RAW = f"dbfs:{RAW_PATH}"                # full DBFS URI for dbutils.fs operations
    CATALOG  = None                               # set to your Unity Catalog name if enabled
    DB_SCHEMA = "bronze"
else:
    # Local development path — project root is two levels above notebooks/
    _here = os.path.dirname(os.path.abspath("__file__"))
    RAW_PATH = os.path.join(_here, "..", "data", "raw")
    RAW_PATH = os.path.normpath(RAW_PATH)
    DB_SCHEMA = "bronze_local"   # use a separate local schema to avoid collision
    print(f"Local raw path: {RAW_PATH}")

INGESTED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Ingestion timestamp: {INGESTED_AT}")

## Cell 2 — Spark Session & Schema Bootstrap

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

if ON_DATABRICKS:
    spark = SparkSession.builder.getOrCreate()
else:
    # Local Spark with Delta support
    spark = (
        SparkSession.builder
        .appName("bronze_ingestion_local")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

spark.conf.set("spark.sql.shuffle.partitions", "8")   # keep small for this dataset size

# ── Create bronze schema (idempotent) ────────────────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB_SCHEMA}")
spark.sql(f"USE {DB_SCHEMA}")
print(f"Active schema: {DB_SCHEMA}")

## Cell 3 — Helper Function

A reusable function that reads a CSV from DBFS/local, appends ingestion metadata,
and writes it as a Delta table. The `MERGE` strategy makes the cell idempotent —
re-running overwrites the table rather than duplicating rows.

In [ ]:
def ingest_csv_to_bronze(
    filename: str,
    table_name: str,
    schema_hint: str = None,
    delimiter: str = ",",
    multiline: bool = False,
) -> None:
    """
    Read a CSV file and write it as a Delta table in the bronze schema.

    Parameters
    ----------
    filename    : CSV filename (inside RAW_PATH)
    table_name  : target Delta table name (without schema prefix)
    schema_hint : optional DDL-style schema string for explicit typing
    delimiter   : CSV field delimiter (default ',')
    multiline   : set True for CSVs with quoted newlines
    """
    full_path = f"{RAW_PATH}/{filename}" if ON_DATABRICKS else os.path.join(RAW_PATH, filename)
    full_table = f"{DB_SCHEMA}.{table_name}"

    reader = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true" if schema_hint is None else "false")
        .option("delimiter", delimiter)
        .option("multiline", str(multiline).lower())
        .option("quote", '"')
        .option("escape", '"')
    )

    if schema_hint:
        reader = reader.schema(schema_hint)

    df = reader.csv(full_path)

    # Append bronze-layer lineage columns
    df = (
        df
        .withColumn("_source_file",  F.lit(filename))
        .withColumn("_ingested_at",  F.lit(INGESTED_AT))
        .withColumn("_bronze_layer", F.lit(True))
    )

    # Overwrite — makes this cell safely re-runnable
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    count = spark.table(full_table).count()
    print(f"[OK] {full_table:55s}  {count:>7,} rows   source: {filename}")

## Cell 4 — Ingest: India Tractor Annual Sales

**Source:** Tractor Manufacturers Association (TMA) of India via Dataful.in  
**Coverage:** FY 2003-04 to FY 2024-25 (22 fiscal years)  
**Key columns:** `fiscal_year`, `year_start`, `year_end`, `total_units_sold`

In [ ]:
INDIA_SALES_SCHEMA = """
    fiscal_year      STRING,
    year_start       INT,
    year_end         INT,
    total_units_sold LONG,
    source           STRING
"""

ingest_csv_to_bronze(
    filename   = "india_tractor_sales_2004_2025.csv",
    table_name = "india_tractor_sales",
    schema_hint = INDIA_SALES_SCHEMA,
)

spark.sql("SELECT * FROM bronze.india_tractor_sales ORDER BY year_start").show(5, truncate=False)

## Cell 5 — Ingest: India Monthly Seasonality

**Source:** TMA derived seasonal distribution  
**Coverage:** 12 months with Kharif/Rabi season labels  
**Key columns:** `month`, `month_name`, `avg_share_pct`, `season_label`

In [ ]:
SEASONALITY_SCHEMA = """
    month         INT,
    month_name    STRING,
    avg_share_pct DOUBLE,
    season_label  STRING,
    notes         STRING
"""

ingest_csv_to_bronze(
    filename    = "india_tractor_monthly_seasonality.csv",
    table_name  = "india_tractor_seasonality",
    schema_hint = SEASONALITY_SCHEMA,
)

spark.sql("SELECT month, month_name, avg_share_pct, season_label FROM bronze.india_tractor_seasonality ORDER BY month").show(truncate=False)

## Cell 6 — Ingest: India Brand Market Share

**Source:** TMA / industry reports  
**Coverage:** Top brands as of 2023  
**Key columns:** `brand`, `parent_company`, `market_share_2023_pct`, `hp_range_focus`

In [ ]:
BRANDS_SCHEMA = """
    brand                 STRING,
    parent_company        STRING,
    country_of_origin     STRING,
    market_share_2023_pct DOUBLE,
    hp_range_focus        STRING,
    key_models            STRING,
    notes                 STRING
"""

ingest_csv_to_bronze(
    filename    = "india_tractor_brand_market_share.csv",
    table_name  = "india_tractor_brands",
    schema_hint = BRANDS_SCHEMA,
)

spark.sql("SELECT brand, market_share_2023_pct, hp_range_focus FROM bronze.india_tractor_brands ORDER BY market_share_2023_pct DESC").show(truncate=False)

## Cell 7 — Ingest: US AEM Tractor Sales by HP Category

**Source:** Association of Equipment Manufacturers (AEM) Farm Equipment Retail Statistics  
**Coverage:** 2010–2025 annual retail units  
**Key columns:** `year`, `under_40hp_units`, `40_to_100hp_units`, `over_100hp_units`, `4wd_units`

In [ ]:
US_AEM_SCHEMA = """
    year               INT,
    under_40hp_units   INT,
    `40_to_100hp_units` INT,
    over_100hp_units   INT,
    `4wd_units`         INT,
    total_2wd_4wd      INT,
    source             STRING
"""

ingest_csv_to_bronze(
    filename    = "us_aem_tractor_sales_by_hp.csv",
    table_name  = "us_aem_tractor_sales_hp",
    schema_hint = US_AEM_SCHEMA,
)

spark.sql("SELECT year, under_40hp_units, `40_to_100hp_units`, over_100hp_units, total_2wd_4wd FROM bronze.us_aem_tractor_sales_hp ORDER BY year").show(5)

## Cell 8 — Ingest: Global Tractor Production by Country

**Source:** TMA, China Tractor Association, AEM, VDMA, JAMA (compiled)  
**Coverage:** 2018–2023 for India, China, USA, Germany, Japan  
**Key columns:** `country`, `year`, `tractors_produced_units`, `tractors_exported_units`

In [ ]:
GLOBAL_PROD_SCHEMA = """
    country                  STRING,
    country_code             STRING,
    year                     INT,
    tractors_produced_units  INT,
    tractors_exported_units  INT,
    domestic_sales_estimate  INT,
    notes                    STRING
"""

ingest_csv_to_bronze(
    filename    = "global_tractor_production_by_country.csv",
    table_name  = "global_tractor_production",
    schema_hint = GLOBAL_PROD_SCHEMA,
)

spark.sql("""
    SELECT country, year, tractors_produced_units, tractors_exported_units
    FROM bronze.global_tractor_production
    ORDER BY year, tractors_produced_units DESC
""").show(10)

## Cell 9 — Ingest: UN Comtrade HS 8701 Trade Flows

**Source:** UN Comtrade API — HS Code 8701 (Tractors)  
**Coverage:** 2015–2022 bilateral export flows  
**Key columns:** `year`, `reporterCode`, `flowCode`, `partnerCode`, `fobValue_usd`, `qty`  

> Note: Country codes follow UN M49 standard. See `config.py` → `COUNTRY_CODES` for mapping.

In [ ]:
COMTRADE_SCHEMA = """
    year            INT,
    reporterCode    INT,
    flowCode        STRING,
    partnerCode     INT,
    netWgt_kg       DOUBLE,
    fobValue_usd    DOUBLE,
    primaryValue_usd DOUBLE,
    qty             INT
"""

ingest_csv_to_bronze(
    filename    = "comtrade_hs8701_tractors_2015_2022.csv",
    table_name  = "comtrade_hs8701_exports",
    schema_hint = COMTRADE_SCHEMA,
)

spark.sql("""
    SELECT year, reporterCode, flowCode, COUNT(*) AS trade_records,
           ROUND(SUM(fobValue_usd) / 1e6, 1) AS total_fob_value_usd_m
    FROM bronze.comtrade_hs8701_exports
    GROUP BY year, reporterCode, flowCode
    ORDER BY year, total_fob_value_usd_m DESC
    LIMIT 10
""").show(truncate=False)

## Cell 10 — Ingest: FRED WPU114 — Farm & Garden Machinery PPI

**Source:** U.S. Bureau of Labor Statistics via FRED  
**Series:** WPU114 — Producer Price Index: Farm & Garden Machinery  
**Coverage:** January 1939 – present (monthly)  
**Key columns:** `observation_date`, `WPU114`

In [ ]:
FRED_WPU114_SCHEMA = """
    observation_date DATE,
    WPU114           DOUBLE
"""

ingest_csv_to_bronze(
    filename    = "fred_wpu114.csv",
    table_name  = "fred_farm_equipment_ppi",
    schema_hint = FRED_WPU114_SCHEMA,
)

spark.sql("""
    SELECT YEAR(observation_date) AS yr, ROUND(AVG(WPU114), 1) AS avg_ppi
    FROM bronze.fred_farm_equipment_ppi
    WHERE YEAR(observation_date) >= 2000
    GROUP BY yr ORDER BY yr
""").show(10)

## Cell 11 — Ingest: FRED WPUFD41312 — Farm Equipment PPI (Detail)

**Source:** U.S. Bureau of Labor Statistics via FRED  
**Series:** WPUFD41312 — PPI: Farm Machinery & Equipment (Final Demand)  
**Coverage:** January 1947 – present (monthly)

In [ ]:
FRED_EQUIP_SCHEMA = """
    observation_date DATE,
    WPUFD41312       DOUBLE
"""

ingest_csv_to_bronze(
    filename    = "fred_farm_equipment_ppi.csv",
    table_name  = "fred_farm_equip_ppi_detail",
    schema_hint = FRED_EQUIP_SCHEMA,
)

spark.sql("""
    SELECT YEAR(observation_date) AS yr, ROUND(AVG(WPUFD41312), 1) AS avg_ppi
    FROM bronze.fred_farm_equip_ppi_detail
    WHERE YEAR(observation_date) >= 2000
    GROUP BY yr ORDER BY yr
""").show(10)

## Cell 12 — Ingest: FRED WPU012202 — Corn Price Index

**Source:** U.S. Bureau of Labor Statistics via FRED  
**Series:** WPU012202 — PPI: Corn  
**Coverage:** January 1971 – present (monthly)

In [ ]:
FRED_CORN_SCHEMA = """
    observation_date DATE,
    WPU012202        DOUBLE
"""

ingest_csv_to_bronze(
    filename    = "fred_corn_price_index.csv",
    table_name  = "fred_corn_price_index",
    schema_hint = FRED_CORN_SCHEMA,
)

spark.sql("""
    SELECT YEAR(observation_date) AS yr, ROUND(AVG(WPU012202), 1) AS avg_corn_ppi
    FROM bronze.fred_corn_price_index
    WHERE YEAR(observation_date) >= 2000
    GROUP BY yr ORDER BY yr
""").show(10)

## Cell 13 — Ingest: FRED WPU012101 — Wheat Price Index

**Source:** U.S. Bureau of Labor Statistics via FRED  
**Series:** WPU012101 — PPI: Wheat  
**Coverage:** July 1991 – present (monthly)

In [ ]:
FRED_WHEAT_SCHEMA = """
    observation_date DATE,
    WPU012101        DOUBLE
"""

ingest_csv_to_bronze(
    filename    = "fred_wheat_price_index.csv",
    table_name  = "fred_wheat_price_index",
    schema_hint = FRED_WHEAT_SCHEMA,
)

spark.sql("""
    SELECT YEAR(observation_date) AS yr, ROUND(AVG(WPU012101), 1) AS avg_wheat_ppi
    FROM bronze.fred_wheat_price_index
    WHERE YEAR(observation_date) >= 2000
    GROUP BY yr ORDER BY yr
""").show(10)

## Cell 14 — Ingest: FRED A368RC1A027NBEA — US Net Farm Income

**Source:** U.S. Bureau of Economic Analysis via FRED  
**Series:** A368RC1A027NBEA — US Net Farm Income (Billions of USD, Annual)  
**Coverage:** 1948 – present (annual)  

> This is a key demand driver: higher farm income → higher tractor purchases.

In [ ]:
FRED_FARM_INCOME_SCHEMA = """
    observation_date    DATE,
    A368RC1A027NBEA     DOUBLE
"""

ingest_csv_to_bronze(
    filename    = "fred_us_net_farm_income.csv",
    table_name  = "fred_us_net_farm_income",
    schema_hint = FRED_FARM_INCOME_SCHEMA,
)

spark.sql("""
    SELECT YEAR(observation_date) AS yr, ROUND(A368RC1A027NBEA, 1) AS net_farm_income_bn_usd
    FROM bronze.fred_us_net_farm_income
    WHERE YEAR(observation_date) >= 2000
    ORDER BY yr
""").show(15)

## Cell 15 — Ingest: FAOSTAT Arable Land (Selected Countries)

**Source:** FAO FAOSTAT — Land Use domain  
**Coverage:** 1961 – present, multiple countries  
**Key columns:** `area` (country name), `year`, `value` (1000 ha of arable land)  
**Note:** FAOSTAT CSV headers contain spaces/parentheses; they are renamed to snake_case on ingest for Delta compatibility.

In [ ]:
# Delta Lake rejects spaces/parentheses in column names — use snake_case.
# CSV header is skipped when schema_hint is set; columns map by position.
FAOSTAT_SCHEMA = """
    area_code      INT,
    area_code_m49  STRING,
    area           STRING,
    item_code      INT,
    item           STRING,
    element_code   INT,
    element        STRING,
    year_code      INT,
    year           INT,
    unit           STRING,
    value          DOUBLE,
    flag           STRING,
    note           STRING
"""

ingest_csv_to_bronze(
    filename    = "faostat_arable_land_selected.csv",
    table_name  = "faostat_arable_land",
    schema_hint = FAOSTAT_SCHEMA,
    multiline   = True,
)

spark.sql("""
    SELECT area, year, ROUND(value, 0) AS arable_land_1000_ha
    FROM bronze.faostat_arable_land
    WHERE year >= 2010 AND area IN ('India', 'China', 'United States of America')
    ORDER BY area, year
""").show(20, truncate=False)

## Cell 16 — Ingest: Supplementary FRED Series (Bonus)

Three additional FRED files present in `data/raw/` that are not in the primary pipeline config  
but may enrich demand-driver analysis:

| Table | FRED Series | Description |
|---|---|---|
| `bronze.fred_net_farm_income_hist` | W007RC1A027NBEA | US net farm income (long-run, 1929–present) |
| `bronze.fred_us_agri_exports_qty` | IQ | US agricultural exports quantity index (monthly) |
| `bronze.fred_us_agri_exports_value` | BOPXGSN | US agricultural exports value in USD bn (quarterly) |


In [ ]:
# W007RC1A027NBEA — long-run US net farm income (annual, back to 1929)
ingest_csv_to_bronze(
    filename    = "fred_net_farm_income_annual.csv",
    table_name  = "fred_net_farm_income_hist",
    schema_hint = "observation_date DATE, W007RC1A027NBEA DOUBLE",
)

# IQ — US agricultural exports quantity index (monthly, many nulls)
ingest_csv_to_bronze(
    filename    = "fred_us_ag_exports.csv",
    table_name  = "fred_us_agri_exports_qty",
    schema_hint = "observation_date DATE, IQ DOUBLE",
)

# BOPXGSN — US agricultural exports value in USD bn (quarterly)
ingest_csv_to_bronze(
    filename    = "fred_us_agri_exports_value.csv",
    table_name  = "fred_us_agri_exports_value",
    schema_hint = "observation_date DATE, BOPXGSN DOUBLE",
)

## Cell 17 — Validation Summary (All 15 Tables)

Quick row-count and schema validation across all bronze tables.

In [ ]:
BRONZE_TABLES = [
    ("bronze.india_tractor_sales",        "India annual tractor sales (FY 2004–2025)"),
    ("bronze.india_tractor_seasonality",  "India monthly seasonal demand share"),
    ("bronze.india_tractor_brands",       "India brand market share 2023"),
    ("bronze.us_aem_tractor_sales_hp",    "US AEM retail sales by HP category"),
    ("bronze.global_tractor_production",  "Global production by country 2018–2023"),
    ("bronze.comtrade_hs8701_exports",    "UN Comtrade HS8701 trade flows 2015–2022"),
    ("bronze.fred_farm_equipment_ppi",    "FRED WPU114 farm equipment PPI (monthly)"),
    ("bronze.fred_farm_equip_ppi_detail", "FRED WPUFD41312 farm equip PPI detail"),
    ("bronze.fred_corn_price_index",      "FRED WPU012202 corn price index (monthly)"),
    ("bronze.fred_wheat_price_index",     "FRED WPU012101 wheat price index (monthly)"),
    ("bronze.fred_us_net_farm_income",    "FRED US net farm income BEA (annual)"),
    ("bronze.faostat_arable_land",        "FAOSTAT arable land by country (1961–present)"),
    # supplementary
    ("bronze.fred_net_farm_income_hist",   "FRED W007RC1A027NBEA US farm income long-run"),
    ("bronze.fred_us_agri_exports_qty",    "FRED IQ US agri exports qty index (monthly)"),
    ("bronze.fred_us_agri_exports_value",  "FRED BOPXGSN US agri exports value USD bn"),
]

print(f"{'TABLE':<40} {'ROWS':>8}  DESCRIPTION")
print("-" * 90)

all_ok = True
for table, desc in BRONZE_TABLES:
    try:
        row_count = spark.table(table).count()
        status = "OK"
    except Exception as exc:
        row_count = -1
        status = f"FAIL: {exc}"
        all_ok = False
    print(f"{table:<40} {row_count:>8,}  {desc}")

print("-" * 90)
print(f"\nBronze layer status: {'ALL OK ✓' if all_ok else 'ERRORS DETECTED — see above'}")

## Cell 18 — Table-Level Metadata (Delta History)

Inspect the last Delta write operation for each table to confirm overwrite timestamps and operation metrics.

In [ ]:
from pyspark.sql.utils import AnalysisException

print("Delta DESCRIBE HISTORY (last operation per table)\n")
for table, _ in BRONZE_TABLES:
    try:
        hist = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1")
        row  = hist.select("version", "timestamp", "operation", "operationMetrics").first()
        print(f"  {table}")
        print(f"    version={row['version']}  ts={row['timestamp']}  op={row['operation']}")
        metrics = row["operationMetrics"]
        if metrics:
            num_files   = metrics.get("numFiles", "?")
            num_output  = metrics.get("numOutputRows", metrics.get("numOutputFiles", "?"))
            print(f"    numFiles={num_files}  numOutputRows={num_output}")
        print()
    except AnalysisException as e:
        print(f"  {table}: {e}\n")

## Cell 19 — Next Steps

Bronze ingestion complete. Proceed to:

**`02_silver_transformation.ipynb`** — Clean, type-cast, and enrich the raw tables:
- Standardise column names to `snake_case`
- Convert fiscal years to calendar years
- Join India sales with seasonality & brand data
- Map UN country codes to country names
- Create unified `silver.global_tractor_sales`

**`03_gold_kpis.ipynb`** — Build analytical aggregates:
- `gold.market_summary` — global volume by year, top-5 countries
- `gold.india_seasonality` — monthly demand with season flags
- `gold.hp_segment_trend` — US HP category mix over time
- `gold.price_vs_sales` — PPI / farm income vs. sales correlation
- `gold.top_exporters` — UN Comtrade top-10 by FOB value
